# Notebook 07 — Compare Genie Spaces: Good vs. Poor Configuration

## Goal

Prove that **curated SQL examples are the difference between a great Genie space and a poor one**. This notebook runs a controlled experiment using 4 hard questions that require multi-table joins, ratio calculations, and domain knowledge.

## The experiment

| | Configured (Good) | No Examples (Poor) |
|---|---|---|
| **Tables** | Same 7 manufacturing tables | Same 7 manufacturing tables |
| **Instructions** | Full business context | Same instructions |
| **Curated Q-to-SQL** | 14 curated Q-to-SQL examples that teach Genie correct joins | **None** |

## 4 Phases

1. **Before** — Run 4 hard questions programmatically against both spaces. The Good space should pass most; the Poor space should fail.
2. **The Fix** — Copy curated Q-to-SQL examples and instructions from the Good space into the Poor space via API.
3. **After** — Re-run the 4 hard questions programmatically against the Poor space (now with examples). Failures should become passes.
4. **Validate in the UI** — Push the 4 hard questions as benchmarks. Run them in the Genie Benchmark tab, review failures, accept knowledge snippets, add a curated example to fix Q4, and iterate until all 4 pass.

## The 4 questions

| # | Question | Why it's hard |
|---|----------|---------------|
| Q1 | Scrap rate for Michigan plants, last 5 days of 2024 | State join + date filter + scrap/units ratio |
| Q2 | Distinct plants with a Maintenance-status production line | Static status filter + join to plants (no date) |
| Q3 | Lines with >5% historical defect rate in 2024 | CASE/HAVING subquery across production_events |
| Q4 | Best-quality shift in Dec 2024 (fewest defects) | events-to-operators join + MIN aggregation |

**Before you start:** Run notebooks **03** (spaces) and **04** (benchmarks).

**Compute:** Serverless.


## Load config and connect to spaces

In [ ]:
%run ./00_workshop_config

In [ ]:
import re
import time
import json
import requests
from datetime import datetime
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
host = w.config.host.rstrip("/")
headers = {**w.config.authenticate(), "Content-Type": "application/json"}

def genie_ui_room_url(space_id):
    m = re.search(r"adb-(\d+)\.", host)
    o = m.group(1) if m else ""
    q = f"?o={o}" if o else ""
    return f"{host}/genie/rooms/{space_id}{q}"

_cfg = spark.table(full_table("workshop_config")).toPandas().set_index("key")["value"].to_dict()
PRIMARY_SPACE_ID = _cfg.get("genie_space_id", "")
NO_EXAMPLES_SPACE_ID = _cfg.get("genie_space_id_configured_no_evals", "")
if not PRIMARY_SPACE_ID:
    raise RuntimeError("genie_space_id not found in workshop_config. Run notebook 03 first.")
if not NO_EXAMPLES_SPACE_ID:
    raise RuntimeError("genie_space_id_configured_no_evals not found. Run notebook 03 first.")

print("=" * 70)
print("SPACES BEING COMPARED (click to open and try manually)")
print("=" * 70)
print(f"  Configured:    {genie_ui_room_url(PRIMARY_SPACE_ID)}")
print(f"  No Examples: {genie_ui_room_url(NO_EXAMPLES_SPACE_ID)}")
print("=" * 70)
print()
print("This notebook sends 4 hard questions to both spaces, then fixes")
print("Space B by adding examples, and re-tests to prove the fix works.")

## Define the shared scoring function

Sends a question to a Genie space, waits for the answer, and compares it to ground truth SQL.

In [ ]:
def _extract_number(text):
    if text is None:
        return None
    nums = re.findall(r"-?\d+(?:\.\d+)?", str(text))
    return float(nums[0]) if nums else None

# NOTE: Genie Conversation API (Public Preview as of April 2026)
def run_benchmarks(benchmarks, space_id, label):
    """Send each benchmark question to Genie and score against ground truth."""
    print(f"\n{label}")
    results = []
    passes = 0
    for i, b in enumerate(benchmarks, 1):
        gt_val = float(spark.sql(b["gt"]).collect()[0][0])
        try:
            start = requests.post(
                f"{host}/api/2.0/genie/spaces/{space_id}/start-conversation",
                headers=headers,
                json={"content": b["q"]},
            )
            if start.status_code != 200:
                print(f"  Q{i}: SKIP (start-conversation {start.status_code})")
                results.append((i, b["q"], gt_val, None, "ERROR"))
                continue
            d = start.json()
            cid, mid = d.get("conversation_id"), d.get("message_id")
            genie_val = None
            status = "FAIL"
            for _ in range(40):
                time.sleep(4)
                poll = requests.get(
                    f"{host}/api/2.0/genie/spaces/{space_id}/conversations/{cid}/messages/{mid}",
                    headers=headers,
                )
                if poll.status_code != 200:
                    continue
                msg = poll.json()
                st = msg.get("status", "")
                if st == "COMPLETED":
                    for att in msg.get("attachments", []):
                        aid = att.get("attachment_id") or att.get("id")
                        if not att.get("query") or not aid:
                            continue
                        qr = requests.get(
                            f"{host}/api/2.0/genie/spaces/{space_id}/conversations/{cid}/messages/{mid}/query-result/{aid}",
                            headers=headers,
                        )
                        if qr.status_code == 200:
                            arr = qr.json().get("statement_response", {}).get("result", {}).get("data_array", [])
                            if arr and arr[0]:
                                genie_val = _extract_number(arr[0][0])
                    if genie_val is not None and gt_val != 0:
                        pct_diff = abs(genie_val - gt_val) / abs(gt_val) * 100
                        status = "PASS" if pct_diff <= BENCHMARK_TOLERANCE_PCT else "FAIL"
                    elif genie_val is not None and gt_val == 0:
                        status = "PASS" if genie_val == 0 else "FAIL"
                    else:
                        status = "FAIL"
                    break
                if st in ("FAILED", "CANCELLED"):
                    status = "FAIL"
                    break
        except Exception as e:
            print(f"  Q{i}: ERROR ({str(e)[:100]})")
            status = "FAIL"
        if status == "PASS":
            passes += 1
        print(f"  Q{i}: {status} (GT={gt_val}, Genie={genie_val})")
        results.append((i, b["q"], gt_val, genie_val, status))
    rate = (passes / len(benchmarks) * 100) if benchmarks else 0
    print(f"  Pass rate: {rate:.1f}% ({passes}/{len(benchmarks)})")
    return results, rate

print("Scoring function ready.")

## Phase 1: "Before" — Ask 4 hard questions to both spaces

These 4 questions require multi-table joins, ratio calculations, and domain knowledge. Space A (with curated Q-to-SQL examples) should pass all 4. No Examples (without examples) should fail on the hardest ones because Genie has to guess the correct joins and formulas.

In [ ]:
hard_questions = [
    {
        "q": "What is the scrap rate for Michigan plants for the last 5 days of 2024? Return the total scrap_count divided by total units_produced as a percentage from quality_metrics_daily.",
        "gt": f"SELECT CAST(ROUND(100.0 * SUM(q.scrap_count) / NULLIF(SUM(q.units_produced), 0), 2) AS DOUBLE) FROM {fqn}.quality_metrics_daily q JOIN {fqn}.plants p ON q.plant_id = p.plant_id WHERE p.state = 'Michigan' AND CAST(q.date AS DATE) >= DATE '2024-12-27' AND CAST(q.date AS DATE) <= DATE '2024-12-31'",
        "short": "Scrap rate Michigan last 5 days",
        "why_hard": "Requires state join + date filter + scrap/units ratio",
    },
    {
        "q": "How many distinct plants have at least one production line with status Maintenance? Return the count of distinct plants.",
        "gt": f"SELECT CAST(COUNT(DISTINCT p.plant_id) AS BIGINT) FROM {fqn}.production_lines pl JOIN {fqn}.plants p ON pl.plant_id = p.plant_id WHERE pl.status = 'Maintenance'",
        "short": "Plants with Maintenance lines",
        "why_hard": "Status filter + join to plants table",
    },
    {
        "q": "How many production lines have a high historical defect rate \u2014 meaning more than 5% of their production_events in 2024 were defect_detected out of total unit_produced events?",
        "gt": f"SELECT CAST(COUNT(*) AS BIGINT) FROM ( SELECT production_line_id, COUNT(CASE WHEN event_type = 'defect_detected' THEN 1 END) AS defects, COUNT(CASE WHEN event_type = 'unit_produced' THEN 1 END) AS produced FROM {fqn}.production_events WHERE YEAR(CAST(event_date AS DATE)) = 2024 GROUP BY production_line_id HAVING produced > 0 AND (100.0 * defects / produced) > 5.0 ) t",
        "short": "Lines with >5% defect rate",
        "why_hard": "CASE/HAVING subquery across production_events",
    },
    {
        "q": "How many defect_detected events did the best-quality shift have in December 2024? Best quality = fewest defects. Join production_events to operators by operator_id to get the shift, group by shift, and return only the lowest defect count.",
        "gt": f"SELECT CAST(MIN(defect_count) AS BIGINT) FROM ( SELECT o.shift, COUNT(*) AS defect_count FROM {fqn}.production_events pe JOIN {fqn}.operators o ON pe.operator_id = o.operator_id WHERE pe.event_type = 'defect_detected' AND CAST(pe.event_date AS DATE) >= DATE '2024-12-01' AND CAST(pe.event_date AS DATE) <= DATE '2024-12-31' GROUP BY o.shift ) t",
        "short": "Best-quality shift Dec 2024",
        "why_hard": "events-to-operators join + MIN aggregation",
    },
]

print("Phase 1: Running 4 hard questions against both spaces...")
print()
for i, hq in enumerate(hard_questions, 1):
    print(f"  Q{i}: {hq['short']}")
    print(f"      Why hard: {hq['why_hard']}")
print()

results_a, rate_a = run_benchmarks(
    hard_questions, PRIMARY_SPACE_ID,
    "=== SPACE A (with curated examples) ==="
)
results_b_before, rate_b_before = run_benchmarks(
    hard_questions, NO_EXAMPLES_SPACE_ID,
    "=== SPACE B (without examples) ==="
)

## Phase 1 results: What failed and why

This cell highlights the differences between the two spaces.

In [ ]:
print("=" * 70)
print("PHASE 1 RESULTS: BEFORE (No Examples space has no curated examples)")
print("=" * 70)
print(f"{'Q#':<4} {'Question':<45} {'Space A':<12} {'No Examples':<12}")
print("-" * 73)
failures_b = []
for ra, rb in zip(results_a, results_b_before):
    qnum, short = ra[0], hard_questions[ra[0]-1]["short"]
    sa_status, sb_status = ra[4], rb[4]
    sa_display = f"PASS ({ra[3]})" if sa_status == "PASS" else f"FAIL ({ra[3]})"
    sb_display = f"PASS ({rb[3]})" if sb_status == "PASS" else f"FAIL ({rb[3]})"
    marker = "  <-- DIFF" if sa_status != sb_status else ""
    print(f"Q{qnum:<3} {short:<45} {sa_display:<12} {sb_display:<12}{marker}")
    if sb_status == "FAIL":
        failures_b.append((qnum, short, hard_questions[qnum-1]["why_hard"], ra[3], rb[3]))
print("-" * 73)
print(f"{'Pass rate:':<49} {rate_a:.0f}%          {rate_b_before:.0f}%")
print()
if failures_b:
    print(f"No Examples space failed {len(failures_b)} question(s):")
    for qnum, short, why, gt, got in failures_b:
        print(f"  Q{qnum} ({short}): expected {gt}, got {got}")
        print(f"       Why: {why}")
    print()
    print("These failures happen because Genie guesses the wrong join paths")
    print("or formulas without curated examples to learn from.")
    print()
    print("Next: Phase 2 will add curated examples to the No Examples space to fix this.")
else:
    print("Both spaces passed all 4 questions. No failures to fix.")
    print("(This can happen if Genie has improved — the experiment still")
    print(" demonstrates best practice: always add curated examples.)")

## Phase 2: Fix the Poor space — add curated examples

Reads the **same template JSON file** that notebook 03 used to create the Good space (`templates/manufacturing_genie_configured.json`), extracts all curated Q-to-SQL examples and business instructions, and PATCHes them onto the Poor space via the `serialized_space` field.


In [ ]:
import uuid, os

# Read the template JSON (same file notebook 03 uses to configure spaces)
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
workshop_dir = "/".join(notebook_path.split("/")[:-1])
# templates/ may be a sibling of the notebook folder or one level up
candidate_paths = [
    f"/Workspace{workshop_dir}/templates/manufacturing_genie_configured.json",
    f"/Workspace{'/'.join(notebook_path.split('/')[:-2])}/templates/manufacturing_genie_configured.json",
]
template_path = next(p for p in candidate_paths if os.path.exists(p))

OLD_FQN = "main.manufacturing_quality_analytics"

def _rewrite_fqn(obj):
    if isinstance(obj, str):
        return obj.replace(OLD_FQN, fqn)
    if isinstance(obj, list):
        return [_rewrite_fqn(x) for x in obj]
    if isinstance(obj, dict):
        return {k: _rewrite_fqn(v) for k, v in obj.items()}
    return obj

with open(template_path) as f:
    genie_template = _rewrite_fqn(json.load(f))

sql_instr = genie_template["sql_instructions"]
curated_raw = genie_template["curated_questions"]

curated_qs = sorted(
    [{"id": uuid.uuid4().hex, "question": [q["question"]], "sql": [q["sql"]]} for q in curated_raw],
    key=lambda x: x["id"],
)

sample_raw = genie_template.get("sample_questions", [])
sample_qs = [{"id": uuid.uuid4().hex, "question": [q]} for q in sample_raw]

TABLE_IDENTIFIERS = sorted([f"{fqn}.{t.split('.')[-1]}" for t in genie_template["table_identifiers"]])
tables_config = [{"identifier": t} for t in TABLE_IDENTIFIERS]

serialized_with_examples = json.dumps({
    "version": 2,
    "config": {"sample_questions": sample_qs},
    "data_sources": {"tables": tables_config},
    "instructions": {
        "text_instructions": [{"content": [sql_instr]}],
        "example_question_sqls": curated_qs,
    },
})

print(f"Loaded template from: {template_path}")
print(f"  {len(curated_qs)} curated Q-to-SQL examples")
print(f"  {len(sql_instr):,} chars of business instructions")
for i, cq in enumerate(curated_qs, 1):
    print(f"  {i:>2}. {cq['question'][0][:70]}")
print()

# PATCH No Examples space with the full config (same mechanism notebook 03 uses)
patch_b = requests.patch(
    f"{host}/api/2.0/genie/spaces/{NO_EXAMPLES_SPACE_ID}",
    headers=headers,
    json={"serialized_space": serialized_with_examples},
)
patch_b.raise_for_status()
print(f"PATCHed No Examples space with {len(curated_qs)} curated examples.")
print("No Examples space now has the same examples and instructions as Configured.")
print()
print("Verify in the UI:")
print(f"  No Examples (patched): {genie_ui_room_url(NO_EXAMPLES_SPACE_ID)}")
print(f"  Configured (reference): {genie_ui_room_url(PRIMARY_SPACE_ID)}")


## Phase 3: Re-run 4 hard questions against the Poor space

Now that the Poor space has curated examples, re-run the exact same 4 hard questions. The questions that failed before should now pass, proving that curated examples were the missing piece.


In [ ]:
print("Phase 3: Re-running 4 hard questions against No Examples space (now with examples)...")

results_b_after, rate_b_after = run_benchmarks(
    hard_questions, NO_EXAMPLES_SPACE_ID,
    "=== SPACE B (after adding curated examples) ==="
)

print()
print("=" * 70)
print("PHASE 3 RESULTS: BEFORE vs. AFTER for No Examples space")
print("=" * 70)
print(f"{'Q#':<4} {'Question':<45} {'Before':<12} {'After':<12}")
print("-" * 73)
fixes = 0
for rb_before, rb_after in zip(results_b_before, results_b_after):
    qnum = rb_before[0]
    short = hard_questions[qnum-1]["short"]
    before_status = rb_before[4]
    after_status = rb_after[4]
    before_display = f"{before_status} ({rb_before[3]})"
    after_display = f"{after_status} ({rb_after[3]})"
    if before_status == "FAIL" and after_status == "PASS":
        marker = "  <-- FIXED"
        fixes += 1
    elif before_status == "FAIL" and after_status == "FAIL":
        marker = "  <-- still failing"
    else:
        marker = ""
    print(f"Q{qnum:<3} {short:<45} {before_display:<12} {after_display:<12}{marker}")
print("-" * 73)
print(f"{'No Examples pass rate:':<49} {rate_b_before:.0f}% -> {rate_b_after:.0f}%")
print()
if fixes > 0:
    print(f"Curated examples fixed {fixes} question(s).")
    print("This proves: examples teach Genie the correct join paths and formulas.")
else:
    print("No new fixes (questions either already passed or still fail).")

## Cleanup: Remove examples from No Examples space

Restores the No Examples space to its original state (no curated examples) so the notebook is idempotent and can be re-run.

In [ ]:
# Rebuild serialized_space WITHOUT examples (same instructions, no curated Q-to-SQL)
serialized_no_examples = json.dumps({
    "version": 2,
    "config": {"sample_questions": []},
    "data_sources": {"tables": tables_config},
    "instructions": {
        "text_instructions": [{"content": [sql_instr]}],
        "example_question_sqls": [],
    },
})

cleanup_patch = requests.patch(
    f"{host}/api/2.0/genie/spaces/{NO_EXAMPLES_SPACE_ID}",
    headers=headers,
    json={"serialized_space": serialized_no_examples},
)
cleanup_patch.raise_for_status()
print("Removed curated examples from No Examples space (restored to original state).")
print("The notebook is idempotent — you can re-run it from the top.")

## Save run history

Appends all three phases of results to `genie_benchmark_runs` so notebook **10** (monitoring) can chart pass rate trends over time.

In [ ]:
ts = datetime.now().isoformat()
sch = StructType([
    StructField("benchmark_id", IntegerType()),
    StructField("question", StringType()),
    StructField("ground_truth", DoubleType()),
    StructField("genie_answer", DoubleType()),
    StructField("status", StringType()),
    StructField("run_timestamp", StringType()),
    StructField("pass_rate", DoubleType()),
    StructField("run_label", StringType()),
])
rows = []
rows += [(r[0], r[1], r[2], r[3], r[4], ts, rate_a, "phase1_space_a_with_examples") for r in results_a]
rows += [(r[0], r[1], r[2], r[3], r[4], ts, rate_b_before, "phase1_space_b_no_examples") for r in results_b_before]
rows += [(r[0], r[1], r[2], r[3], r[4], ts, rate_b_after, "phase3_space_b_after_fix") for r in results_b_after]

tbl = f"{fqn}.genie_benchmark_runs"
try:
    spark.createDataFrame(rows, sch).write.mode("append").saveAsTable(tbl)
except Exception:
    spark.sql(f"DROP TABLE IF EXISTS {tbl}")
    spark.createDataFrame(rows, sch).write.saveAsTable(tbl)
print(f"Saved {len(rows)} results to {tbl}")

## Final summary

In [ ]:
print("=" * 70)
print("EXPERIMENT SUMMARY")
print("=" * 70)
print()
print(f"{'Q#':<4} {'Question':<40} {'Space A':<10} {'B Before':<10} {'B After':<10}")
print("-" * 74)
for ra, rb_b, rb_a in zip(results_a, results_b_before, results_b_after):
    qnum = ra[0]
    short = hard_questions[qnum-1]["short"]
    print(f"Q{qnum:<3} {short:<40} {ra[4]:<10} {rb_b[4]:<10} {rb_a[4]:<10}")
print("-" * 74)
print(f"{'Pass rate:':<44} {rate_a:.0f}%        {rate_b_before:.0f}%        {rate_b_after:.0f}%")
print()
print("CONCLUSION:")
if rate_b_before < rate_a and rate_b_after >= rate_a:
    diff = int(rate_a - rate_b_before)
    print(f"  Without curated examples, No Examples space failed {diff}% of hard questions.")
    print(f"  After adding the same curated examples, No Examples matched Configured at {rate_b_after:.0f}%.")
    print(f"  Curated Q-to-SQL examples are what make a Genie space great.")
elif rate_b_before == rate_a:
    print("  Both spaces scored equally — Genie handled these questions without examples.")
    print("  Curated examples are still best practice for production reliability.")
else:
    print(f"  No Examples went from {rate_b_before:.0f}% to {rate_b_after:.0f}% after adding examples.")
    print(f"  Some questions remain hard even with examples — these need refined prompts.")
print()
print("Open both spaces to try it yourself:")
print(f"  Configured: {genie_ui_room_url(PRIMARY_SPACE_ID)}")
print(f"  No Examples: {genie_ui_room_url(NO_EXAMPLES_SPACE_ID)}")
print()
print("Next: Notebook 08 — Security and governance (column masking).")

## Phase 4: Validate with UI benchmarks

The programmatic comparison above uses numeric tolerance to score answers. The Genie **Benchmark** tab compares full SQL result sets and is the most accurate evaluator.

Push the 4 hard questions as benchmarks to the **Good** space and validate in the UI.

### Step 1: Add the 4 questions as benchmarks

1. Open the **Configured** space link printed above
2. Click the **Benchmark** tab (top nav)
3. Add each question below with its ground-truth SQL

| # | Question | Ground Truth SQL |
|---|----------|------------------|
| Q1 | What is the scrap rate for Michigan plants for the last 5 days of 2024? Return scrap_count / units_produced as a % from quality_metrics_daily. | `SELECT ROUND(100.0 * SUM(q.scrap_count) / NULLIF(SUM(q.units_produced), 0), 2) FROM CATALOG.SCHEMA.quality_metrics_daily q JOIN CATALOG.SCHEMA.plants p ON q.plant_id = p.plant_id WHERE p.state = 'Michigan' AND q.date >= '2024-12-27' AND q.date <= '2024-12-31'` |
| Q2 | How many distinct plants have at least one production line with status Maintenance? | `SELECT COUNT(DISTINCT p.plant_id) FROM CATALOG.SCHEMA.production_lines pl JOIN CATALOG.SCHEMA.plants p ON pl.plant_id = p.plant_id WHERE pl.status = 'Maintenance'` |
| Q3 | How many production lines have a defect rate above 5% in 2024 (defect_detected / unit_produced events)? | `SELECT COUNT(*) FROM (SELECT production_line_id FROM CATALOG.SCHEMA.production_events WHERE YEAR(CAST(event_date AS DATE)) = 2024 GROUP BY production_line_id HAVING 100.0 * SUM(CASE WHEN event_type='defect_detected' THEN 1 ELSE 0 END) / NULLIF(SUM(CASE WHEN event_type='unit_produced' THEN 1 ELSE 0 END),0) > 5) t` |
| Q4 | How many defect_detected events did the best-quality shift have in Dec 2024? Best = fewest defects. Join production_events to operators, group by shift, return lowest count. | `SELECT MIN(defect_count) FROM (SELECT o.shift, COUNT(*) AS defect_count FROM CATALOG.SCHEMA.production_events pe JOIN CATALOG.SCHEMA.operators o ON pe.operator_id = o.operator_id WHERE pe.event_type='defect_detected' AND pe.event_date >= '2024-12-01' AND pe.event_date <= '2024-12-31' GROUP BY o.shift) t` |

Replace `CATALOG.SCHEMA` with your actual catalog and schema (e.g., `gm_ama_demos.genie_workshop_manufacturing`).

### Step 2: Run benchmarks

1. Click **Start new run** in the Benchmark tab
2. Review failures — click each failed question to see the failure analysis
3. **Accept knowledge snippets** where they are correct (these are suggestions — use your SME judgment to decide which patterns Genie should learn)
4. **Update ground truth** if Genie's SQL is equivalent but styled differently (e.g., `ROUND(AVG(...) * 100)` vs `CAST(ROUND(..., 0) AS BIGINT)`)
5. **Re-run** until Q1-Q3 pass


### Step 3: Fix Q4 by adding a curated example

Q4 (best-quality shift in Dec 2024) likely still fails because Genie doesn't have a curated example that combines **shift join + date filter + scalar MIN aggregation**.

Add a curated Q-to-SQL example that teaches this pattern using a **different time range** (Q3 2024), so Q4 (Dec 2024) still tests generalization:

1. Click **Configure** (top nav)
2. Scroll to **SQL Queries and Functions**
3. Click **Add** to add a new curated example
4. Paste the question and SQL below:

**Question:**
```
How many defect_detected events did the worst-performing shift have in Q3 2024? Worst = most defects. Join production_events to operators by operator_id, group by shift, return only the highest count.
```

**SQL** (replace `CATALOG.SCHEMA` with your actual catalog and schema):
```sql
SELECT MAX(defect_count) AS answer
FROM (
  SELECT o.shift, COUNT(*) AS defect_count
  FROM CATALOG.SCHEMA.production_events pe
  JOIN CATALOG.SCHEMA.operators o ON pe.operator_id = o.operator_id
  WHERE pe.event_type = 'defect_detected'
    AND CAST(pe.event_date AS DATE) >= DATE '2024-07-01'
    AND CAST(pe.event_date AS DATE) < DATE '2024-10-01'
  GROUP BY o.shift
) t
```

5. Click **Save**

This teaches Genie: events-to-operators join + date range filter + scalar aggregation of grouped shift counts. Q4 uses MIN instead of MAX and December instead of Q3 — same pattern, different parameters.


### Step 4: Re-run benchmarks

1. Go back to the **Benchmark** tab
2. Click **Start new run**
3. All 4 questions should now pass
4. If any still fail, review the failure analysis, accept snippets, and iterate

**Target:** 100% on all 4 hard questions.

Once all 4 pass in the UI, you have proven that curated examples + knowledge snippets make a Genie space production-ready.

**Next:** Notebook 08 — Security and governance (column masking).
